In [0]:
spark.version

In [0]:
%run ../tests/test_silver_layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import logging

In [0]:
# Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("Weather Silver Layer")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("bronze_weather_table", "")
dbutils.widgets.text("silver_weather", "")

bronze_weather_table = dbutils.widgets.get("bronze_weather_table") or "chicago_taxi_data.bronze.bronze_weather"
silver_weather_data = dbutils.widgets.get("silver_weather") or "chicago_taxi_data.silver.silver_weather"


In [0]:
df_bronze_weather = spark.read.format("delta").table(bronze_weather_table)

In [0]:
df_bronze_weather.printSchema()

In [0]:
display(df_bronze_weather.limit(5))

In [0]:
df_silver_weather = df_bronze_weather.select(
    F.to_date("date").alias("date"),
    F.col("hour").cast("int").alias("hour"),
    F.round(F.col("temp")).cast("int").alias("temperature"),
    F.round(F.col("wind_speed")).cast("int").alias("wind_speed"),
    F.round(F.col("precip"), 1).cast("decimal(10,1)").alias("precipitation"),
    F.col("humidity").cast("int").alias("humidity")
)

In [0]:
df_silver_weather = df_silver_weather.dropDuplicates(subset=["date", "hour"])
df_silver_weather = df_silver_weather.withColumn("year", F.year("date"))
window_spec = Window.partitionBy("year").orderBy("date", "hour")

df_silver_weather = (
    df_silver_weather
    .withColumn("temperature", F.last("temperature", ignorenulls=True).over(window_spec))
    .withColumn("wind_speed", F.last("wind_speed", ignorenulls=True).over(window_spec))
)

In [0]:
df_silver_weather.printSchema()

In [0]:
display(df_silver_weather.limit(5))

In [0]:

required_columns = ["date", "hour", "temperature", "humidity", "precipitation", "wind_speed"]

try:
    logger.info("Starting Weather DQ checks Silver Layer")

    validate_no_nulls(df_silver_weather, "date")
    validate_no_nulls(df_silver_weather, "hour")
    validate_schema(df_silver_weather, required_columns)
    validate_duplicates(df_silver_weather, ["date", "hour"])
    validate_temperature_range(df_silver_weather, "temperature", -40, 50)
    
    logger.info("Silver Weather DQ tests Passed.")
except Exception as e:
    logger.error(f"DQ Failed: {str(e)}")
    dbutils.notebook.exit(str(e))

In [0]:
try:
    logger.info(f"Write df_silver_weather to `{silver_weather_data}` Gold Layer")
    df_silver_weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_weather_data)
    logger.info(f"df_silver_weather saved successfully to `{silver_weather_data}`")
except Exception as e:
    logger.error(f"Failed to Save: {e}")
    dbutils.notebook.exit(f"Failed to Save: {e}")
